# Model Evaluation - Accuracy & Confusion Matrix

This notebook evaluates the Main Agent decisions (continue / schedule / end) against the labeled conversations in `sms_conversations.json`.

For every labeled recruiter turn, we rebuild the conversation history up to that point, ask the Main Agent what action it would take, and compare the prediction to the true label.

## Setup

In [ ]:
import sys
import os
import json

sys.path.append(os.path.abspath(os.path.join("..", "app", "modules")))

from main_agent import decide_action

## Load the labeled conversations

In [ ]:
with open(os.path.join("..", "data", "sms_conversations.json"), "r", encoding="utf-8") as f:
    conversations = json.load(f)

print("number of conversations:", len(conversations))

## Build the evaluation examples

Each example is the conversation history before a labeled recruiter turn, together with the true label.

The first turn of every conversation is skipped because there is no history yet to decide on.

In [ ]:
examples = []

for conv in conversations:
    history = ""
    for turn in conv["turns"]:
        if turn["speaker"] == "recruiter" and turn["label"] is not None and history != "":
            examples.append({"history": history.strip(), "label": turn["label"]})
        history = history + turn["speaker"] + ": " + turn["text"] + "\n"

print("number of evaluation examples:", len(examples))

## Run the Main Agent on every example

This cell calls the OpenAI API for every example, so it takes a few minutes.

In [ ]:
predictions = []
true_labels = []

for i, ex in enumerate(examples):
    pred = decide_action(ex["history"])
    predictions.append(pred)
    true_labels.append(ex["label"])
    print(i + 1, "/", len(examples), "true:", ex["label"], "| predicted:", pred)

## Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(true_labels, predictions)
print("Accuracy:", round(accuracy, 3))

## Confusion Matrix

The rows are the true labels and the columns are the predictions. The diagonal is the correct decisions.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

labels = ["continue", "schedule", "end"]
cm = confusion_matrix(true_labels, predictions, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()
plt.title("Main Agent - Confusion Matrix")
plt.show()

## Show mistakes

In [ ]:
for ex, pred in zip(examples, predictions):
    if pred != ex["label"]:
        print("true:", ex["label"], "| predicted:", pred)
        print(ex["history"])
        print("-" * 60)